# Customer Retention Intelligence — Streamlit Dashboard (Phase 12)

## Objective

Demonstrate the Phase 12 Streamlit dashboard's data flow **without duplicating any of it**:

```
Business User -> Streamlit Dashboard -> HTTP -> FastAPI (Phase 11)
    -> MLflow registered model (Phase 10) -> SHAP (Phase 8) / Next-Best-Action (Phase 9)
```

This notebook does not implement, predict, explain, or recommend anything
itself. It exercises `dashboard/api_client.py` (the same module the
Streamlit app uses) against a **real, running FastAPI server**, and shows
how a real feature row becomes a request payload.

**Before running this notebook:** start the API in a separate terminal:

```bash
uvicorn api.main:app --reload
```

The dashboard itself is launched separately with:

```bash
streamlit run dashboard/app.py
```

which cannot run inside a notebook cell (it's a persistent server
process) -- this notebook instead documents and verifies the same
HTTP contract the dashboard relies on.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import JSON, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dashboard import api_client

PROJECT_ROOT

PosixPath('/Users/dhananjaymandal/customer-retention-intelligence')

## 1. Check API Health

Uses `dashboard.api_client.get_health` -- the exact function the
dashboard's sidebar calls.

In [2]:
health = api_client.get_health()
print(f"ok={health.ok}, status_code={health.status_code}")
display(JSON(health.data or {"error": health.error}))

if not health.ok:
    print(
        "\nFastAPI does not appear to be running. Start it with:\n"
        "    uvicorn api.main:app --reload\n"
        "then re-run this cell."
    )

ok=True, status_code=200


<IPython.core.display.JSON object>

## 2. Check Model Info

In [3]:
model_info = api_client.get_model_info()
print(f"ok={model_info.ok}, status_code={model_info.status_code}")
display(JSON(model_info.data or {"error": model_info.error}))

ok=True, status_code=200


<IPython.core.display.JSON object>

## 3. Build a Request Payload From a Real Feature Row

`api_client.build_customer_payload` is a pure data-formatting function
(NaN -> null, numpy -> native types, Timestamp -> ISO date string) -- it
does not predict, impute, or invent anything. It's the exact function the
dashboard calls after a user picks a customer/snapshot in the sidebar.

In [4]:
features_df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "customer_features.parquet")
sample_row = features_df.sample(n=1, random_state=99).iloc[0]

payload = api_client.build_customer_payload(sample_row)
print(f"customer_id: {payload['customer_id']}")
print(f"snapshot_date: {payload['snapshot_date']}")
print(f"Total fields in payload: {len(payload)}")
print(f"Target column excluded: {'churn_next_30_days' not in payload}")

customer_id: C006391
snapshot_date: 2025-01-31
Total fields in payload: 133
Target column excluded: True


## 4. Call POST /customer-analysis

The same call the dashboard's "Analyze Customer" button makes.

In [5]:
if health.ok:
    analysis = api_client.customer_analysis(payload, top_k=5)
    print(f"ok={analysis.ok}, status_code={analysis.status_code}")
    if analysis.ok:
        display(JSON(analysis.data))
    else:
        print(f"Error: {analysis.error}")
else:
    print("Skipped -- API is not reachable (see Section 1).")

ok=True, status_code=200


<IPython.core.display.JSON object>

## 5. Demonstrate Graceful Error Handling

Point the client at a port nothing is listening on and confirm
`api_client` returns a clear, structured failure -- never an unhandled
exception -- exactly what lets the dashboard stay usable when the
backend is down.

In [6]:
import os

original_base_url = os.environ.get("API_BASE_URL")
os.environ["API_BASE_URL"] = "http://127.0.0.1:59999"  # nothing is listening here

unreachable_result = api_client.get_health()
print(f"ok={unreachable_result.ok}")
print(f"error={unreachable_result.error}")

if original_base_url is None:
    os.environ.pop("API_BASE_URL", None)
else:
    os.environ["API_BASE_URL"] = original_base_url

ok=False
error=Could not connect to the API at http://127.0.0.1:59999. Start it with: uvicorn api.main:app --reload


## Summary

- The dashboard's HTTP client (`dashboard/api_client.py`) was exercised against the real, running FastAPI service.
- The request payload came from a **real row** in `customer_features.parquet`, converted by a pure formatting function -- no fabricated feature values.
- Connection failures are handled gracefully with a clear message, never a raw exception.
- **No prediction, SHAP computation, or recommendation logic was run inside this notebook or the dashboard** -- FastAPI remains the single source of truth for all three.

To see the full interactive dashboard:

```bash
uvicorn api.main:app --reload
streamlit run dashboard/app.py
```